<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

# Introduction 
It is finally time to try building the whole pipeline. Note that it is a draft, tehre are many thihngs that could be done better and it is by no means a final product. Given that, the data we will be using is from lightcurvelynx. I will also choose the parameters based on what they used in their tutorial, which may be not at all representative of the true AGN population. The point of this notebook is not to get it perfect, but to design the architecture that can then be applied to real data. 

In [1]:
import os
import matplotlib.pyplot as plt
from warnings import filters
import jax.numpy as jnp
import jax
from caskade import Param, forward
import numpy as np
import pandas as pd
from tinygp import GaussianProcess, kernels
import jaxopt
from astropy.cosmology import Planck18
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.redshift import RedshiftDistFunc
from lightcurvelynx.base_models import FunctionNode
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableRADECSampler
from lightcurvelynx.math_nodes.scipy_random import SamplePDF
from lightcurvelynx.models.agn import AGN
from lightcurvelynx.obstable.opsim import OpSim
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.utils.plotting import plot_lightcurves
from lightcurvelynx.survey_info import SurveyInfo

/home/zoe/cosmographi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Obtaining the data
We will start off by trying to get all the data we need from lightcurvelynx. Again, I will be using some paramter distirbution that may not represent the true AGN population. I will, however, keep  wavelength constant and redshift at 0.

In [2]:
# TODO: Look at interplay between GP params, wavelength and redshift. 
# copy-pasted from tiny_gp_experiments.ipynb:

passband_group = PassbandGroup.from_preset(
    preset="LSST",
)
# It will take a while to download it. It took 9 min on Helen
obstable = OpSim.from_url(
    "https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs4.3/baseline/baseline_v4.3.5_10yrs.db",
)

#chekc out https://lightcurvelynx.readthedocs.io/en/latest/notebooks/pre_executed/agn.html#AGN-Damped-Random-Walk-Example

lg_bh_mass = NumpyRandomFunc("uniform", low=7.0, high=9.0)

#I'll just keep this instead of caskade because I'm only gathering some preliminaruy data, will not have it written in final code
bh_mass = FunctionNode(
    lambda lg_mass: 10**lg_mass,
    lg_mass=lg_bh_mass,
    node_label="bh_mass",
)


def edd_ratio_pdf(value):
    xi = 10**-1.65
    lambda_br = 10**-1.84
    delta1 = 0.471 - 0.7
    delta2 = 2.53
    min_lambda = 0.01
    max_lambda = 1.0
    value = np.asarray(value)
    fill_mask = (value >= min_lambda) & (value <= max_lambda)
    prob = np.zeros_like(value)
    prob[fill_mask] = xi / (
        (value[fill_mask] / lambda_br) ** delta1 + (value[fill_mask] / lambda_br) ** delta2
    )
    return prob

edd_ratio = SamplePDF(edd_ratio_pdf)

radec = ObsTableRADECSampler(
    obstable,
    radius=3.0,  # degrees
    node_label="ra_dec_sampler",
)

model = AGN(
    t0=obstable.time_bounds()[0],
    redshift=0.1,
    cosmology=Planck18,
    passband_group=passband_group,
    redshift_dist_func=0,
    blackhole_mass=bh_mass,
    ra=radec.ra,
    dec=radec.dec,
    edd_ratio=edd_ratio,
)

survey_info = SurveyInfo(obstable=obstable, passband_group=passband_group)

rng = np.random.default_rng(42)

df = simulate_lightcurves(
    model=model,
    num_samples=100,
    survey_info=survey_info,
    param_cols=[
        "bh_mass.lg_mass",
        "AGN_0.edd_ratio"
    ],       
    rng=rng)
 


Simulating:   0%|          | 0/100 [00:00<?, ?obj/s]

Simulating:   1%|          | 1/100 [00:00<00:19,  5.15obj/s]

Simulating:   2%|▏         | 2/100 [00:00<00:17,  5.63obj/s]

Simulating:   3%|▎         | 3/100 [00:00<00:16,  5.80obj/s]

Simulating:   4%|▍         | 4/100 [00:00<00:16,  5.88obj/s]

...

Simulating:  97%|█████████▋| 97/100 [00:19<00:00,  4.92obj/s]

Simulating:  98%|█████████▊| 98/100 [00:19<00:00,  5.11obj/s]

Simulating:  99%|█████████▉| 99/100 [00:19<00:00,  5.23obj/s]

Simulating: 100%|██████████| 100/100 [00:19<00:00,  5.42obj/s]

Simulating: 100%|██████████| 100/100 [00:19<00:00,  5.04obj/s]

In [3]:
print(df.columns)
print(df.loc[3, "params"])

Index(['id', 'ra', 'dec', 'nobs', 't0', 'z', 'bh_mass_lg_mass',
       'AGN_0_edd_ratio', 'lightcurve', 'params'],
      dtype='object')
{'NumpyRandomFunc:integers_2.low': 0, 'NumpyRandomFunc:integers_2.high': 2019461, 'NumpyRandomFunc:integers_2.function_node_result': 886297, 'ra_dec_sampler.selected_table_index': 886297, 'ra_dec_sampler.ra': 291.0277783348556, 'ra_dec_sampler.dec': -52.86164407132714, 'ra_dec_sampler.time': 62639.25852381482, 'AGN_0.ra': 291.0277783348556, 'AGN_0.dec': -52.86164407132714, 'AGN_0.redshift': 0.1, 'AGN_0.t0': 60980.00158187724, 'AGN_0.distance': 475822267.5121877, 'AGN_0.blackhole_mass': 31309715.827585578, 'AGN_0.edd_ratio': 0.022993851538879465, 'AGN_0.inclination_rad': 0.27194858522637905, 'AGN_0.blackhole_mass_gram': 6.225654800032216e+40, 'AGN_0.critical_accretion_rate': 4.383360215861981e+25, 'AGN_0.blackhole_accretion_rate': 1.0079033404496103e+24, 'AGN_0.bolometric_luminosity': 9.071130064046493e+43, 'AGN_0.mag_i': -19.894153485017398, 'AGN_0.sf

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
data = [["id", "bh_mass_lg_mass", "AGN_0_edd_ratio", "flux_perfect", "mu avg.", "mu sd", "var avg.", "var sd."]]

raw_vals = []

# data = jnp.asarray(df.loc[0, "lightcurve"][["mjd", "flux_perfect"]].to_numpy())

def build_gp_var_fixed(theta: tuple[Param], x: jnp.ndarray) -> GaussianProcess:
    """ Build a Gaussian Process with the given parameters and input data.
    Precondition: theta is a tuple of (log_sigma, log_scale) parameters.
    """
    log_sigma, log_scale = theta
    kernel = kernels.quasisep.Exp(scale=jnp.exp(log_scale), sigma=jnp.exp(log_sigma))
    return GaussianProcess(kernel, x, diag=0.02)

def neg_log_likelihood_var_fixed(theta, X, y):
    gp = build_gp_var_fixed(theta, X)
    return -gp.log_probability(y)

solver = jaxopt.ScipyMinimize(fun=neg_log_likelihood_var_fixed)

scale = Param("scale", value=float(jnp.log(100.0)), shape=(), description="scale parameter")
sigma = Param("sigma", value=float(jnp.log(1.01)), shape=(), description="sigma parameter")
theta = (jnp.log(float(sigma.value)), jnp.log(float(scale.value)))

for i in range(100):
    lightcurve = df.loc[i, "lightcurve"]
    if lightcurve is None or "flux_perfect" not in lightcurve.columns:
        print(f"Warning: Sample {i} has no lightcurve data or missing 'flux_perfect' column. Skipping.")
        continue
    sample_data = jnp.asarray(df.loc[i, "lightcurve"][["mjd", "flux_perfect"]].to_numpy())
    x = sample_data[:, 0]
    y = sample_data[:, 1]
    y_mean = jnp.mean(y)
    y_stand = y/y_mean
    if jnp.any(y_stand <= 0):
        print(f"Warning: Sample {i} has non-positive standardized flux values. Clipping.")
        y_stand = jnp.clip(y_stand, 0, None)
    if jnp.any(jnp.isnan(y_stand)):
        print(f"Warning: Sample {i} has NaN standardized flux values. Skipping.")
        continue
    y_log = jnp.log(y_stand)
    soln = solver.run(theta, X=x, y=y_log)
    gp = build_gp_var_fixed(soln.params, x)
    cond_gp = gp.condition(y_log).gp
    mu, var = jnp.exp(cond_gp.loc), jnp.exp(cond_gp.variance)
    raw_vals.append([mu, var])
    data.append([
        df.loc[i, "id"],
        df.loc[i,"bh_mass_lg_mass"],
        df.loc[i, "AGN_0_edd_ratio"],
        df.loc[i, "lightcurve"]["flux_perfect"].mean(),
        float(mu.mean()),
        float(mu.std()),
        float(var.mean()),
        float(var.std()),
    ])


2026-05-27 11:46:32.847980: W external/xla/xla/service/gpu/nvptx_compiler.cc:760] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.9.86). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


E0527 11:47:44.716779  399700 pjrt_stream_executor_client.cc:2826] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to get module function:CUDA_ERROR_OUT_OF_MEMORY: out of memory


XlaRuntimeError: RESOURCE_EXHAUSTED: Failed to get module function:CUDA_ERROR_OUT_OF_MEMORY: out of memory

In [ ]:
data = np.asarray(data)
print(data)
pd.DataFrame(data).to_csv("agn_gp_results.csv", index=False)

In [ ]:
print(raw_vals)
pd.DataFrame(raw_vals).to_csv("agn_gp_raw_vals.csv", index=False)